# Pipeline de Entrenamiento

_Cerrar el ciclo: del feature store al modelo de regresion entrenado y evaluado, con consistencia entrenamiento-serving._

**Modulo 1 — Feature Engineering & Feature Stores** | DSRP Machine Learning Engineering
**Profesor:** Miguel Arquez

![Pipeline de Entrenamiento](assets/header.png)


## Introduccion

En el Notebook 3 definimos features en **Feast** y aprendimos a materializarlas y
servirlas. Pero un feature store no es un fin en si mismo: existe para
**alimentar modelos**. En este notebook corremos el ciclo **end-to-end**, usando
las **dos mitades** del feature store:

```
                 OFFLINE                                      ONLINE
   get_historical_features (point-in-time)      get_online_features (Redis, ms)
                    |                                           |
                    v                                           v
            set de entrenamiento  ->  modelo entrenado  ->  inferencia en serving
```

Es decir:

1. Construimos el **set de entrenamiento** con `get_historical_features`:
   para cada casa pedimos las features **en su propio `event_timestamp`**
   (point-in-time: nunca un valor "del futuro").
2. Entrenamos un **regresor** que predice `SalePrice` (sobre `log1p(SalePrice)`,
   como en la competencia de Kaggle) y lo **evaluamos** (RMSE, MAE, R²).
3. **Materializamos** el ultimo valor de cada feature a Redis y simulamos
   **peticiones de inferencia** con `get_online_features`.
4. Verificamos la **consistencia entrenamiento-serving**: las features que sirve
   Redis son las mismas con las que se entreno el modelo.

> **La idea grande: consistencia entrenamiento-serving.** El set de entrenamiento
> sale de las *mismas* definiciones de features que luego sirven online. Por eso
> el modelo ve en produccion exactamente la misma representacion con la que
> aprendio: no hay sorpresas de *skew*. El feature store es el contrato que une
> ambos mundos.

> **Requisitos.** Este notebook corre de verdad contra la plataforma (sin
> fallbacks silenciosos): necesitas Redis arriba y el CLI de feast instalado.
>
> ```bash
> cd ../platform && docker compose up -d --build     # redis + feast-ui + airflow
> pip install "feast[redis]" redis                    # en el entorno del notebook
> ```

## Paso 0 — setup y chequeo de la plataforma

Definimos rutas y verificamos que la plataforma este lista **antes** de empezar:
Redis respondiendo en `localhost:6379` y el CLI de `feast` instalado. Si algo
falta, el notebook falla aqui con instrucciones — mejor un error claro al inicio
que un fallback silencioso a mitad de camino.

In [ ]:
import os
import shutil
import socket
import subprocess
from datetime import datetime, timedelta, timezone

import numpy as np
import pandas as pd

REPO = os.path.abspath(os.path.join("..", "platform", "feature_repo"))
PARQUET_PATH = os.path.join(REPO, "data", "housing_features.parquet")
REDIS_HOST, REDIS_PORT = "localhost", 6379


def redis_arriba() -> bool:
    """True si el online store (Redis) responde en localhost:6379."""
    try:
        with socket.create_connection((REDIS_HOST, REDIS_PORT), timeout=2):
            return True
    except OSError:
        return False


assert shutil.which("feast") is not None, (
    'Falta el CLI de feast. Instala:  pip install "feast[redis]" redis'
)
assert redis_arriba(), (
    "Redis no responde en localhost:6379. Levanta la plataforma:\n"
    "    cd ../platform && docker compose up -d --build"
)

def feast_cli(*args):
    """Corre un comando del CLI de feast dentro del repo, mostrando la salida."""
    res = subprocess.run(["feast", *args], cwd=REPO, capture_output=True, text=True)
    print(res.stdout or "", res.stderr or "")
    assert res.returncode == 0, f"feast {' '.join(args)} fallo (exit {res.returncode})"
    return res

print("repo de features:", REPO)
print("plataforma OK: redis en localhost:6379 y CLI de feast disponible")

### Bootstrap del offline store

El offline store (el parquet de `feature_repo/data/`) normalmente lo deja listo
el **Notebook 3** (o el DAG de Airflow). Si no existe, lo construimos aqui mismo
desde el CSV de Ames — con **timestamps escalonados en 30 dias** (cada casa
"ocurrio" en un momento distinto), que es lo que hace interesante el join
point-in-time del paso siguiente. Cerramos con `feast apply` para asegurar que
el registry este al dia.

In [ ]:
# Mapa de calidad ordinal: Po<Fa<TA<Gd<Ex; NaN/None = 0 (no existe esa parte).
QUAL_MAP = {"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5}


def construir_parquet_offline():
    """(Re)construye el parquet del offline store desde el CSV de Ames (como en NB3)."""
    candidatos = [os.environ.get("HOUSING_CSV"),
                  os.path.join("..", "..", "data", "housing_train.csv")]
    csv = next((p for p in candidatos if p and os.path.exists(p)), None)
    assert csv, "No encontre housing_train.csv (esperado en data/ de la raiz del repo)."
    df = pd.read_csv(csv)

    feat = pd.DataFrame()
    feat["house_id"] = df["Id"].astype("int64")
    feat["overall_qual"] = df["OverallQual"].astype("int64")
    feat["gr_liv_area"] = df["GrLivArea"].astype("float32")
    feat["total_bsmt_sf"] = df["TotalBsmtSF"].fillna(0).astype("float32")
    feat["first_flr_sf"] = df["1stFlrSF"].astype("float32")
    feat["garage_cars"] = df["GarageCars"].fillna(0).astype("int64")
    feat["garage_area"] = df["GarageArea"].fillna(0).astype("float32")
    feat["year_built"] = df["YearBuilt"].astype("int64")
    feat["lot_area"] = df["LotArea"].astype("float32")
    feat["full_bath"] = df["FullBath"].astype("int64")
    feat["neighborhood"] = df["Neighborhood"].fillna("None").astype(str)
    feat["exter_qual_ord"] = (df["ExterQual"].fillna("None")
                              .map(QUAL_MAP).fillna(0).astype("int64"))
    feat["sale_price"] = df["SalePrice"].astype("float32")

    # Timestamps escalonados: cada casa "ocurre" en un dia distinto de los
    # ultimos 30. Asi el offline store tiene una dimension temporal real.
    now = datetime.now(timezone.utc)
    feat["event_timestamp"] = [now - timedelta(days=int(i % 30)) for i in range(len(feat))]
    feat["created"] = pd.Timestamp(now)

    os.makedirs(os.path.dirname(PARQUET_PATH), exist_ok=True)
    feat.to_parquet(PARQUET_PATH, index=False)
    print(f"[bootstrap] {len(feat)} filas -> {PARQUET_PATH}")


if os.path.exists(PARQUET_PATH):
    print(f"[bootstrap] el parquet ya existe (Notebook 3); lo reutilizo: {PARQUET_PATH}")
else:
    construir_parquet_offline()

# Registra (o confirma) entidades y feature views en el registry de Feast.
feast_cli("apply")

## Paso 1 — features HISTORICAS: el set de entrenamiento point-in-time

La forma *correcta* de armar un set de entrenamiento es con
**`get_historical_features`**: le pasas un **dataframe de entidades** — QUE casas
y EN QUE `event_timestamp` — y Feast hace el **join point-in-time** contra el
offline store: para cada fila devuelve el ultimo valor de cada feature
**anterior o igual** a ese timestamp. Nunca un valor del futuro.

Aqui pedimos cada casa **en su propio momento** (el `event_timestamp` de su
fila, que en un caso real seria el momento de la venta — el momento de la
etiqueta). Eso es lo que evita el **data leakage temporal**: el modelo solo ve
lo que se sabia *cuando ocurrio la etiqueta*.

> El mismo mecanismo, en miniatura y con el efecto visible a simple vista, esta
> en el DAG `hello_historical_features` de la plataforma: el mismo usuario
> pedido en 3 fechas devuelve 3 valores distintos de `clicks`.

In [ ]:
from feast import FeatureStore

store = FeatureStore(repo_path=REPO)

FEATURES = [
    "house_features:overall_qual",
    "house_features:gr_liv_area",
    "house_features:total_bsmt_sf",
    "house_features:first_flr_sf",
    "house_features:garage_cars",
    "house_features:garage_area",
    "house_features:year_built",
    "house_features:lot_area",
    "house_features:full_bath",
    "house_features:neighborhood",
    "house_features:exter_qual_ord",
]

# Dataframe de entidades: cada casa pedida EN SU PROPIO momento (el timestamp
# de su fila = el momento de la "venta", donde nace la etiqueta).
base = pd.read_parquet(PARQUET_PATH)
entity_df = base[["house_id", "event_timestamp"]].copy()

train_df = store.get_historical_features(
    entity_df=entity_df, features=FEATURES,
).to_df()

# El objetivo `sale_price` viaja en el parquet (no es una feature servida).
train_df = train_df.merge(base[["house_id", "sale_price"]], on="house_id")
print(f"set de entrenamiento point-in-time: {train_df.shape}")
train_df.head()

## Paso 2 — entrenar un regresor

Predecimos `SalePrice` con un **`RandomForestRegressor`** dentro de un `Pipeline`
de sklearn. Entrenamos sobre **`log1p(SalePrice)`** (asi el modelo optimiza el error
en escala logaritmica, que es la metrica de Kaggle y reduce el peso de las casas
caras). El pipeline encapsula el preprocesamiento (one-hot del barrio) junto con el
modelo, de modo que *exactamente los mismos pasos* corren en entrenamiento y en
serving. Eso es, de nuevo, consistencia entrenamiento-serving, ahora a nivel de
codigo del modelo.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

numeric = ["overall_qual", "gr_liv_area", "total_bsmt_sf", "first_flr_sf",
           "garage_cars", "garage_area", "year_built", "lot_area",
           "full_bath", "exter_qual_ord"]
categorical = ["neighborhood"]

X = train_df[numeric + categorical].copy()
y = train_df["sale_price"].astype(float)
y_log = np.log1p(y)                       # entrenamos en escala logaritmica

X_train, X_test, y_train, y_test, ylog_train, ylog_test = train_test_split(
    X, y, y_log, test_size=0.25, random_state=42,
)

pre = ColumnTransformer(
    transformers=[("cat", OneHotEncoder(handle_unknown="ignore"), categorical)],
    remainder="passthrough",
)
model = Pipeline(steps=[
    ("pre", pre),
    ("reg", RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)),
])

model.fit(X_train, ylog_train)            # objetivo = log1p(SalePrice)
print("Modelo de regresion entrenado. Features:", numeric + categorical)

## Paso 3 — evaluar

Medimos las metricas tipicas de regresion sobre el precio en su escala original
(en dolares): **RMSE** (raiz del error cuadratico medio), **MAE** (error absoluto
medio) y **R²** (fraccion de varianza explicada). Ademas reportamos el **RMSE sobre
`log(SalePrice)`**, que es la **metrica oficial de la competencia de Kaggle** (y la
que el modelo optimiza directamente). Cerramos con un grafico de **predicho vs
real**.

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Predecimos en escala log y volvemos a dolares con expm1.
pred_log = model.predict(X_test)
pred = np.expm1(pred_log)

rmse = np.sqrt(mean_squared_error(y_test, pred))
mae = mean_absolute_error(y_test, pred)
r2 = r2_score(y_test, pred)
rmse_log = np.sqrt(mean_squared_error(ylog_test, pred_log))   # metrica de Kaggle

print(f"RMSE  (USD)          : {rmse:,.0f}")
print(f"MAE   (USD)          : {mae:,.0f}")
print(f"R2                   : {r2:.4f}")
print(f"RMSE sobre log(price): {rmse_log:.4f}   <- metrica oficial de Kaggle")

In [ ]:
import matplotlib.pyplot as plt

# Predicho vs real: cuanto mas pegados a la diagonal, mejor.
fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.scatter(y_test, pred, s=14, alpha=0.5, c="#4c78a8")
lims = [min(y_test.min(), pred.min()), max(y_test.max(), pred.max())]
ax.plot(lims, lims, "--", c="#e45756", lw=1.5, label="prediccion perfecta")
ax.set_xlabel("SalePrice real"); ax.set_ylabel("SalePrice predicho")
ax.set_title(f"Predicho vs real (R²={r2:.3f})")
ax.legend(); plt.tight_layout(); plt.show()

## Paso 4 — persistir el modelo

Antes de servir predicciones, **persistimos el modelo entrenado** en disco para
reutilizarlo (lo cargaremos tal cual en el paso de serving, como haria un servicio
de inferencia):

In [ ]:
import joblib

MODEL_PATH = os.path.join(REPO, "data", "housing_model.joblib")
joblib.dump(model, MODEL_PATH)
print(f"Modelo guardado en: {MODEL_PATH}")
print(f"Metricas -> RMSE={rmse:,.0f}  MAE={mae:,.0f}  R2={r2:.4f}  RMSE_log={rmse_log:.4f}")

## Paso 5 — features ONLINE: materializar y servir la inferencia

Entrenamos con features **historicas** (`get_historical_features`). En
**produccion**, cuando llega una peticion —"¿cuanto vale *esta* casa?"— no
recalculamos todo el historico. El flujo online tiene dos partes:

1. **`feast materialize-incremental`** copia el **ultimo** valor de cada feature
   del offline store (parquet) al **online store** (Redis). Es un paso batch,
   programado (en la plataforma lo hace el DAG de Airflow).
2. **`get_online_features`** lee esos valores desde Redis en milisegundos y se
   los pasa al modelo — la misma llamada que haria un servicio de inferencia.

```
   peticion (house_id) -> get_online_features (Redis) -> modelo.predict -> SalePrice
```

Y el cierre del circulo: verificamos que **las features que sirve Redis son
exactamente las mismas** que las del set de entrenamiento. Esa es la
consistencia entrenamiento-serving que promete el feature store.

In [ ]:
# 1) offline -> online: carga el ULTIMO valor de cada feature a Redis.
#    Usamos `materialize` con ventana explicita (los ultimos 31 dias cubren
#    todos los timestamps escalonados del parquet). El `-incremental` que usa el
#    DAG arranca desde la ultima marca de agua y podria saltarse casas si ya se
#    materializo hoy (p. ej. desde el Notebook 3).
now = datetime.now(timezone.utc)
start = (now - timedelta(days=31)).strftime("%Y-%m-%dT%H:%M:%S")
end = now.strftime("%Y-%m-%dT%H:%M:%S")
feast_cli("materialize", start, end)

# 2) Cargamos el modelo persistido, como lo haria un servicio de inferencia.
served_model = joblib.load(MODEL_PATH)

# Simulamos peticiones de serving para algunas casas.
example_ids = [int(h) for h in train_df["house_id"].head(3)]
refs = [f"house_features:{c}" for c in (numeric + categorical)]

online = store.get_online_features(
    features=refs, entity_rows=[{"house_id": h} for h in example_ids],
).to_dict()

# to_dict() devuelve columnas por nombre (sin el prefijo "house_features:").
X_online = pd.DataFrame({c: online[c] for c in (numeric + categorical)})
assert X_online.notna().all().all(), (
    "El online store devolvio nulos: revisa que `feast materialize` haya corrido."
)

# El modelo es un Pipeline: aplica el MISMO preprocesamiento del entrenamiento.
pred_online = np.expm1(served_model.predict(X_online))    # de log a dolares
for h, p in zip(online["house_id"], pred_online):
    print(f"  house_id={h:<5d} -> SalePrice predicho: ${p:,.0f}")

# 3) Consistencia entrenamiento-serving: lo que sirve Redis == lo que vio el
#    modelo al entrenar (el offline store tiene una fila por casa, asi que el
#    "ultimo" valor online es el mismo del set de entrenamiento).
h0 = example_ids[0]
fila_train = train_df.loc[train_df["house_id"] == h0, numeric + categorical].iloc[0]
fila_online = X_online.iloc[online["house_id"].index(h0)]
assert np.allclose(fila_train[numeric].astype(float), fila_online[numeric].astype(float)), \
    "skew! las features online difieren de las de entrenamiento"
assert fila_train["neighborhood"] == fila_online["neighborhood"]
print(f"\nOK consistencia train-serving: house_id={h0} recibe online las mismas "
      f"features con las que se entreno.")

## Repaso — el ciclo cerrado

Acabamos de recorrer el ciclo completo de un sistema de ML del mundo real,
usando las **dos mitades** del feature store:

1. **Feature store (Feast)** define y sirve features de forma consistente.
2. **HISTORICO**: `get_historical_features` armo el **set de entrenamiento**
   point-in-time correcto — cada casa pedida en su propio `event_timestamp`,
   sin leakage temporal.
3. Un **modelo de regresion** se entreno con ese set, dentro de un pipeline que
   garantiza que el preprocesamiento sera identico en serving.
4. El modelo se **evaluo** (RMSE, MAE, R² y el RMSE sobre log(price) de Kaggle)
   y se **persistio** en disco.
5. **ONLINE**: `feast materialize` cargo el ultimo valor de cada feature a Redis
   y `get_online_features` sirvio la inferencia en tiempo real.
6. Verificamos la **consistencia entrenamiento-serving**: las features online
   son exactamente las del set de entrenamiento.

**Por que importa la consistencia entrenamiento-serving:** el modelo aprende sobre
una representacion concreta de los datos. Si en produccion las features se
calculan de otra forma (otra mediana de imputacion, otro escalado, otra version de
la transformacion), el modelo recibe entradas "fuera de distribucion" y se degrada
en silencio. El feature store + el pipeline de sklearn son las dos piezas que
garantizan que *lo que se entrena es lo que se sirve*.

**Orquestacion.** Este mismo ciclo lo automatizan los DAGs de **Apache Airflow**
de la plataforma compartida (`platform/dags/`):

| DAG | Que demuestra |
|---|---|
| `hello_feature_engineering` | El ciclo **online** en miniatura (5 usuarios → Redis → serving) |
| `hello_historical_features` | El ciclo **offline** en miniatura (point-in-time visible a simple vista) |
| `feature_engineering_pipeline` | El pipeline real de housing: `extract → transform → validate → apply → materialize → train` |

Airflow es el **orquestador compartido del curso**, introducido en este modulo.